In [ ]:
'Cellule 1'
from google.colab import drive

drive.mount('/content/drive')

import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'numpy',
    'pandas',
    'matplotlib',
    'scikit-learn',
    'torch',
])


In [ ]:
'Cellule 2'
from pathlib import Path
from types import SimpleNamespace
import sys

CFG = SimpleNamespace(
    project_dir=Path('/content/drive/MyDrive/New project/python'),
    processed_env_dir=Path('/content/drive/MyDrive/New project/python/processed_env_real_csv'),
    output_dir=Path('/content/drive/MyDrive/New project/python/enhanced_runs'),
    states=['arkansas', 'california'],
    configs=['baseline', 'climate', 'soil', 'topography', 'all'],
    epochs=250,
    batch_size=32,
    lr=1e-3,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.1,
    model_dim=96,
    num_heads=6,
    num_transformer_layers=2,
    gru_hidden_dim=64,
    static_hidden_dim=32,
    ffn_expansion=4,
    temporal_mask_rate=0.15,
    reconstruction_weight=0.2,
    class_weight_power=0.5,
    sampler_power=0.5,
    label_smoothing=0.05,
    gradient_clip_norm=1.0,
    patience=30,
    early_stopping_patience=30,
    seed=2021,
    use_env_covariates=True,
    use_weighted_sampler=True,
    cpu=False,
)

CFG.output_dir.mkdir(parents=True, exist_ok=True)

if not CFG.project_dir.exists():
    raise FileNotFoundError(f'project_dir introuvable: {CFG.project_dir}')

if not CFG.processed_env_dir.exists():
    raise FileNotFoundError(f'processed_env_dir introuvable: {CFG.processed_env_dir}')

if str(CFG.project_dir) not in sys.path:
    sys.path.insert(0, str(CFG.project_dir))

print('project_dir:', CFG.project_dir)
print('processed_env_dir:', CFG.processed_env_dir)
print('output_dir:', CFG.output_dir)


In [ ]:
'Cellule 3'
from dataset_enhanced import prepare_enhanced_splits, resolve_dataset_paths
from enhanced_mctnet import EnhancedMCTNet
from train_enhanced_model import run_full_ablation, set_seed, plot_ablation_comparison

for state in CFG.states:
    npz_path, json_path = resolve_dataset_paths(CFG.processed_env_dir, state)
    print(f'\n[{state}]')
    print('npz:', npz_path)
    print('json:', json_path)

    prepared_baseline = prepare_enhanced_splits(
        npz_path=npz_path,
        json_path=json_path,
        config_name='baseline',
        use_env_covariates=CFG.use_env_covariates,
    )
    print('  baseline x_train:', prepared_baseline.x_train.shape)
    print('  baseline valid_mask_train:', prepared_baseline.valid_mask_train.shape)
    print('  baseline y_train:', prepared_baseline.y_train.shape)
    print('  baseline dynamic_env_train:', prepared_baseline.dynamic_env_train.shape)
    print('  baseline static_env_train:', prepared_baseline.static_env_train.shape)
    print('  baseline temporal_input_dim:', prepared_baseline.temporal_input_dim)
    print('  baseline static_input_dim:', prepared_baseline.static_input_dim)
    print('  class_names:', prepared_baseline.class_names)

    prepared_all = prepare_enhanced_splits(
        npz_path=npz_path,
        json_path=json_path,
        config_name='all',
        use_env_covariates=CFG.use_env_covariates,
    )
    print('  all dynamic features:', prepared_all.dynamic_feature_names)
    print('  all static features:', prepared_all.static_feature_names)
    print('  all temporal_input_dim:', prepared_all.temporal_input_dim)
    print('  all static_input_dim:', prepared_all.static_input_dim)

    model = EnhancedMCTNet(
        temporal_input_dim=prepared_all.temporal_input_dim,
        static_input_dim=prepared_all.static_input_dim,
        num_classes=len(prepared_all.class_names),
        seq_len=prepared_all.x_train.shape[1],
        model_dim=CFG.model_dim,
        num_heads=CFG.num_heads,
        num_transformer_layers=CFG.num_transformer_layers,
        gru_hidden_dim=CFG.gru_hidden_dim,
        static_hidden_dim=CFG.static_hidden_dim,
        dropout=CFG.dropout,
        ffn_expansion=CFG.ffn_expansion,
    )
    print('  model created:', model.__class__.__name__)


In [ ]:
'Cellule 4'
set_seed(CFG.seed)
results = run_full_ablation(
    processed_env_dir=CFG.processed_env_dir,
    output_dir=CFG.output_dir,
    states=CFG.states,
    args=CFG,
)

print(results)


In [ ]:
'Cellule 5'
from IPython.display import Image as IPImage, display

for state in CFG.states:
    for config in CFG.configs:
        png = CFG.output_dir / state / f'learning_curves_enhanced_{config}.png'
        if png.exists():
            print(state, config, png.name)
            display(IPImage(filename=str(png)))


In [ ]:
'Cellule 6'
from IPython.display import Image as IPImage, display

for state in CFG.states:
    for config in CFG.configs:
        png = CFG.output_dir / state / f'confusion_matrix_enhanced_{config}.png'
        if png.exists():
            print(state, config, png.name)
            display(IPImage(filename=str(png)))


In [ ]:
'Cellule 7'
import pandas as pd

summary_path = CFG.output_dir / 'enhanced_ablation_summary.csv'
mctnet_baseline = {
    'arkansas': {'OA': 0.968, 'Kappa': 0.951, 'F1': 0.933},
    'california': {'OA': 0.852, 'Kappa': 0.806, 'F1': 0.829},
}

if not summary_path.exists():
    print(f'Fichier de synthese introuvable: {summary_path}')
    print('Lance d abord la cellule 4 pour generer les resultats enhanced.')
else:
    summary = pd.read_csv(summary_path)
    if summary.empty:
        print(f'Le fichier {summary_path.name} est vide.')
    else:
        summary['state'] = pd.Categorical(summary['state'], categories=CFG.states, ordered=True)
        summary['config'] = pd.Categorical(summary['config'], categories=CFG.configs, ordered=True)
        summary = summary.sort_values(['state', 'config']).reset_index(drop=True)

        expected = pd.MultiIndex.from_product([CFG.states, CFG.configs], names=['state', 'config']).to_frame(index=False)
        progress = expected.merge(summary[['state', 'config']], on=['state', 'config'], how='left', indicator=True)
        progress['status'] = progress['_merge'].map({'both': 'done', 'left_only': 'pending'})
        progress = progress[['state', 'config', 'status']]

        print('Results available')
        display(summary)

        print('Experiment progress')
        display(progress)

        oa_pivot = summary.pivot_table(index='config', columns='state', values='OA', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)
        kappa_pivot = summary.pivot_table(index='config', columns='state', values='Kappa', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)
        f1_pivot = summary.pivot_table(index='config', columns='state', values='F1_macro', aggfunc='first').reindex(index=CFG.configs, columns=CFG.states)

        print('OA pivot')
        display(oa_pivot)
        print('Kappa pivot')
        display(kappa_pivot)
        print('F1 pivot')
        display(f1_pivot)

        improvement_rows = []
        for _, row in summary.iterrows():
            state = str(row['state'])
            if state not in mctnet_baseline:
                continue
            base = mctnet_baseline[state]
            improvement_rows.append({
                'state': state,
                'config': str(row['config']),
                'delta_OA': float(row['OA']) - float(base['OA']),
                'delta_Kappa': float(row['Kappa']) - float(base['Kappa']),
                'delta_F1': float(row['F1_macro']) - float(base['F1']),
            })

        improvements = pd.DataFrame(improvement_rows)
        if improvements.empty:
            print('No improvements to compare yet.')
        else:
            improvements['state'] = pd.Categorical(improvements['state'], categories=CFG.states, ordered=True)
            improvements['config'] = pd.Categorical(improvements['config'], categories=CFG.configs, ordered=True)
            improvements = improvements.sort_values(['state', 'config']).reset_index(drop=True)
            print('Improvements vs original MCTNet')
            display(improvements)

        best_by_state = summary.sort_values(['state', 'F1_macro'], ascending=[True, False]).groupby('state', as_index=False).head(1)
        print('Best available configuration per state by F1_macro')
        display(best_by_state[['state', 'config', 'OA', 'Kappa', 'F1_macro', 'best_epoch', 'temporal_input_dim', 'static_input_dim']])


In [ ]:
'Cellule 8'
from IPython.display import Image as IPImage, display

summary_path = CFG.output_dir / 'enhanced_ablation_summary.csv'
barplot_path = CFG.output_dir / 'enhanced_comparison_barplot.png'

if not summary_path.exists():
    print(f'Fichier de synthese introuvable: {summary_path}')
    print('Lance d abord la cellule 4 pour generer les resultats enhanced.')
else:
    plot_ablation_comparison(
        summary_csv=summary_path,
        output_dir=CFG.output_dir,
    )
    if barplot_path.exists():
        display(IPImage(filename=str(barplot_path)))
    else:
        print(f'Barplot non genere: {barplot_path}')
